In [11]:
from datasets import load_dataset
import requests

HF_TOKEN="REDACTED_HF_TOKEN"
corpus_ds = load_dataset("Tim-Pinecone/sec-10k-qa", "corpus", token=HF_TOKEN)
questions_ds = load_dataset("Tim-Pinecone/sec-10k-qa", "questions", token=HF_TOKEN)

# Build a lookup: corpus index -> document_id string
doc_ids = [row["document_id"] for row in corpus_ds["train"]]

# Filter questions for Apple (AAPL) filings only
filtered = [
    row for row in questions_ds["train"]
    if "AAPL" in doc_ids[row["document_id"]]
]

# Get unique AAPL corpus document indices
aapl_doc_indices = sorted(set(row["document_id"] for row in filtered))

# Build corpus: list of full filing texts for AAPL
corpus = [corpus_ds["train"][i]["text"] for i in aapl_doc_indices]

print(f"Total questions: {len(questions_ds['train'])}")
print(f"AAPL questions: {len(filtered)}")
print(f"AAPL documents: {len(corpus)}")

# Fetch AAPL filing metadata from SEC EDGAR to get exact PDF filenames
headers = {"User-Agent": "filings evaluation script"}
submissions = requests.get(
    "https://data.sec.gov/submissions/CIK0000320193.json",
    headers=headers,
).json()

# Build accession -> primary document lookup
recent = submissions["filings"]["recent"]
accession_to_doc = {}
for acc, form, primary_doc in zip(
    recent["accessionNumber"], recent["form"], recent["primaryDocument"]
):
    accession_to_doc[acc] = (form, primary_doc)

print("\nAAPL filings in dataset (with direct PDF links):")
for i in aapl_doc_indices:
    doc_id_str = doc_ids[i]
    parts = doc_id_str.split("_")
    accession = parts[3]
    acc_no_dashes = accession.replace("-", "")
    form_type, primary_doc = accession_to_doc.get(accession, ("?", "?"))
    url = f"https://www.sec.gov/Archives/edgar/data/320193/{acc_no_dashes}/{primary_doc}"
    print(f"  {parts[1]} {form_type} -> {accession}")
    print(f"    Download: {url}")

questions = [row["question"] for row in filtered]
passages  = [row["chunk_must_contain"] for row in filtered]

Total questions: 950
AAPL questions: 50
AAPL documents: 5

AAPL filings in dataset (with direct PDF links):
  AAPL 10-K -> 0000320193-21-000105
    Download: https://www.sec.gov/Archives/edgar/data/320193/000032019321000105/aapl-20210925.htm
  AAPL 10-K -> 0000320193-22-000108
    Download: https://www.sec.gov/Archives/edgar/data/320193/000032019322000108/aapl-20220924.htm
  AAPL 10-K -> 0000320193-23-000106
    Download: https://www.sec.gov/Archives/edgar/data/320193/000032019323000106/aapl-20230930.htm
  AAPL 10-K -> 0000320193-24-000123
    Download: https://www.sec.gov/Archives/edgar/data/320193/000032019324000123/aapl-20240928.htm
  AAPL 10-K -> 0000320193-25-000079
    Download: https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm


In [9]:
from mtcb.evaluators.simple import SimpleEvaluator
from chonkie import RecursiveChunker

evaluator = SimpleEvaluator(
    corpus=corpus,
    questions=questions,
    relevant_passages=passages,
    chunker=RecursiveChunker(chunk_size=1000),
    embedding_model="openai/text-embedding-3-small",
)

result = evaluator.evaluate(k=[1, 3, 5, 10])
print(result)

# Access individual metrics
print("\n--- Metrics Summary ---")
for metric_name, k_values in result.metrics.items():
    for k, score in k_values.items():
        print(f"{metric_name}@{k}: {score:.4f}")

Evaluating with k values: [1, 3, 5, 10]
Using tokenizer: gpt2
Loading question embeddings...


Embedding batch failed at index 0: Invalid input: Model 'openai/text-embedding-3-small' not found in catalog. Use 'provider:model' format (e.g., 'openai:text-embedding-3-small') or check available models with list_models()
Chunking all documents...


Chunking: 100%|██████████| 5/5 [00:00<00:00, 112.61it/s, total_chunks=1471]


Created 1471 chunks in 0.05s
Embedding all chunks (batch_size=128)...


Embedding batch failed at index 0: Invalid input: Model 'openai/text-embedding-3-small' not found in catalog. Use 'provider:model' format (e.g., 'openai:text-embedding-3-small') or check available models with list_models()
Embedding batch failed at index 128: Invalid input: Model 'openai/text-embedding-3-small' not found in catalog. Use 'provider:model' format (e.g., 'openai:text-embedding-3-small') or check available models with list_models()
Embedding batch failed at index 256: Invalid input: Model 'openai/text-embedding-3-small' not found in catalog. Use 'provider:model' format (e.g., 'openai:text-embedding-3-small') or check available models with list_models()
Embedding batch failed at index 384: Invalid input: Model 'openai/text-embedding-3-small' not found in catalog. Use 'provider:model' format (e.g., 'openai:text-embedding-3-small') or check available models with list_models()
Embedding batch failed at index 512: Invalid input: Model 'openai/text-embedding-3-small' not found in

Questions:   0%|          | 0/50 [00:00<?, ?it/s]


IndexError: list index out of range